# Reference solution — Модуль 3 (House Prices)

> **Этот ноутбук — эталонное решение для преподавателя.** Он не
> предназначен для студентов первой итерации: их workflow —
> [`notebook.ipynb`](notebook.ipynb) с пустыми `TODO`. Этот файл
> нужен, чтобы вы (или ассистент) могли:
> - провести семинар, показав готовый прогон от начала до конца;
> - проверить, что окружение в Colab/Kaggle/локально живо и даёт
  ожидаемые цифры;
> - сверять решения студентов с эталоном при проверке ДЗ.

**Ожидаемые цифры** (Colab CPU, seed=42):

- Ridge + OneHot: RMSE ≈ 0.16—0.18 на log(SalePrice).
- CatBoost: RMSE ≈ 0.125—0.130, обучение 20—40 секунд.
- Топ-3 SHAP-фичи: `OverallQual`, `GrLivArea`, что-то из (`Neighborhood`, `TotalBsmtSF`, `YearBuilt`).
- Самый дорогой дом в тесте: обычно ~$400—500k (предсказание модели).
- На графике «линия vs функция» Ridge — строго прямая, CatBoost — ломаная со ступеньками и плато на больших площадях.

## Шаг 0. Установка библиотек

In [ ]:
!pip install -q catboost shap && echo "[ok] catboost и shap установлены"

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f'pandas {pd.__version__} | numpy {np.__version__} | окружение готово')

## Шаг 1. Данные — House Prices (Ames Housing)

1 460 домов в городе Эймс, штат Айова, 80 признаков, таргет — `SalePrice` в долларах.

### Подключение датасета (для семинара)

Код в следующей ячейке ищет `train.csv` тремя способами:

1. **Kaggle с подключённым competition** — рекомендуется. Подключается через `Add Input → Competitions → "House Prices"`, потом обязательно `Run → Restart Session` (иначе kernel не увидит файлы).
2. **OpenML через `fetch_openml`** — для Colab/локально. На Kaggle тоже работает с Internet=On, но иногда отваливается по 504.
3. Если оба не сработали — RuntimeError с понятной инструкцией.

**Совет на семинар:** покажите студентам оба пути и явно скажите про `Restart Session` — это самая частая ловушка. Заодно поучите их, что Kaggle-input живёт по двум разным путям (`/kaggle/input/<slug>/` или `/kaggle/input/competitions/<slug>/`), и `glob.glob('/kaggle/input/**/train.csv', recursive=True)` — нормальный способ их разруливать.

In [ ]:
import glob
from pathlib import Path

# Универсальный поиск train.csv в /kaggle/input/ — ловит и плоский,
# и вложенный (/competitions/) layout.
kaggle_candidates = glob.glob('/kaggle/input/**/train.csv', recursive=True)
kaggle_candidates = [c for c in kaggle_candidates if 'house' in c.lower()]

if kaggle_candidates:
    df = pd.read_csv(kaggle_candidates[0])
    print(f'[ok] Загружено из Kaggle Input: {kaggle_candidates[0]} ({df.shape})')
else:
    try:
        ames = fetch_openml(name='house_prices', version=1, as_frame=True, parser='auto')
        df = ames.frame.copy()
        print(f'[ok] Загружено через OpenML: {df.shape}')
    except Exception as e:
        raise RuntimeError(
            'На Kaggle подключите competition (Add Input → Competitions → '
            '"House Prices") и сделайте Restart Session. Либо включите '
            f'Internet. Исходная ошибка OpenML: {e}'
        )

if 'Id' in df.columns:
    df = df.drop(columns=['Id'])

print(df.shape)
df[['LotArea', 'YearBuilt', 'OverallQual', 'Neighborhood', 'GrLivArea', 'SalePrice']].head()

In [ ]:
y = np.log1p(df.pop('SalePrice'))
X = df

cat_features = X.select_dtypes(include='object').columns.tolist()
num_features = X.select_dtypes(exclude='object').columns.tolist()
print(f'категориальных: {len(cat_features)}')
print(f'числовых: {len(num_features)}')
print(f'таргет log(SalePrice): mean={y.mean():.2f}, std={y.std():.2f}')

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

## Шаг 2. Бейзлайн — Ridge с one-hot

In [ ]:
X_tr_fill = X_tr.copy()
X_te_fill = X_te.copy()
for c in num_features:
    med = X_tr_fill[c].median()
    X_tr_fill[c] = X_tr_fill[c].fillna(med)
    X_te_fill[c] = X_te_fill[c].fillna(med)
for c in cat_features:
    X_tr_fill[c] = X_tr_fill[c].fillna('missing').astype(str)
    X_te_fill[c] = X_te_fill[c].fillna('missing').astype(str)

preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
    ('num', StandardScaler(), num_features),
])
ridge = Pipeline([
    ('prep', preprocess),
    ('reg', Ridge(alpha=10.0, random_state=RANDOM_STATE)),
])

t0 = time.time()
ridge.fit(X_tr_fill, y_tr)
ridge_time = time.time() - t0

ridge_rmse = mean_squared_error(y_te, ridge.predict(X_te_fill)) ** 0.5
print(f'Ridge  RMSE = {ridge_rmse:.4f}  ({ridge_time:.1f} s)')

## Шаг 3. TODO 1 — CatBoost (решение)

Гиперпараметры подобраны так, чтобы стабильно бить порог `RMSE ≤ 0.135` на seed=42. Если хочется выжать ещё, попробуйте `iterations=3000`, `learning_rate=0.02`, `l2_leaf_reg=3—10`.

In [ ]:
from catboost import CatBoostRegressor

X_tr_cb = X_tr.copy()
X_te_cb = X_te.copy()
for c in cat_features:
    X_tr_cb[c] = X_tr_cb[c].fillna('missing').astype(str)
    X_te_cb[c] = X_te_cb[c].fillna('missing').astype(str)

cat_model = CatBoostRegressor(
    iterations=1500,
    learning_rate=0.03,
    depth=6,
    cat_features=cat_features,
    eval_metric='RMSE',
    early_stopping_rounds=50,
    random_seed=RANDOM_STATE,
    verbose=200,
)
t0 = time.time()
cat_model.fit(X_tr_cb, y_tr, eval_set=(X_te_cb, y_te))
cat_time = time.time() - t0
cat_rmse = mean_squared_error(y_te, cat_model.predict(X_te_cb)) ** 0.5

print(f'CatBoost RMSE = {cat_rmse:.4f}  ({cat_time:.1f} s)')
assert cat_rmse <= 0.135, f'RMSE {cat_rmse:.4f} выше 0.135 — проверь cat_features, iterations и early_stopping_rounds'

In [ ]:
comparison = pd.DataFrame([
    {'model': 'Ridge + OneHot', 'RMSE (log price)': round(ridge_rmse, 4), 'time, s': round(ridge_time, 2)},
    {'model': 'CatBoost',       'RMSE (log price)': round(cat_rmse, 4),   'time, s': round(cat_time, 2)},
])
comparison.sort_values('RMSE (log price)')

## Шаг 4. Сравнение моделей графически (решение)

Три картинки для семинара — каждая отвечает на свой вопрос, и каждая срабатывает у студентов лучше, чем число в табличке.

1. **Predicted vs Actual** — насколько модель попадает в цену. Облако CatBoost явно плотнее к диагонали.
2. **Линия vs функция** — главный визуал лекции. Ridge рисует прямую (по математике линейной модели), CatBoost — ломаную со ступеньками.
3. **Training curve** — видно, как `early_stopping_rounds` ловит момент, когда eval-RMSE перестал улучшаться.

In [ ]:
# График 1: предсказание vs реальность
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True, sharex=True)

ridge_preds = ridge.predict(X_te_fill)
cb_preds = cat_model.predict(X_te_cb)

for ax, name, preds in [
    (axes[0], 'Ridge (линейка)', ridge_preds),
    (axes[1], 'CatBoost (ансамбль деревьев)', cb_preds),
]:
    ax.scatter(y_te, preds, alpha=0.4, s=20)
    lo, hi = float(y_te.min()), float(y_te.max())
    ax.plot([lo, hi], [lo, hi], 'r--', linewidth=1.2, label='идеальное предсказание')
    rmse = mean_squared_error(y_te, preds) ** 0.5
    ax.set_title(f'{name}\nRMSE = {rmse:.4f}')
    ax.set_xlabel('настоящая log(цена)')
    ax.legend(loc='upper left')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('предсказанная log(цена)')
plt.suptitle('Чем плотнее точки к красной диагонали — тем точнее модель', y=1.02)
plt.tight_layout()
plt.savefig('predicted_vs_actual.png', dpi=120, bbox_inches='tight')
plt.show()
print('[ok] сохранил predicted_vs_actual.png')

In [ ]:
# График 2: «линия vs функция» — фиксируем все фичи на типичных значениях,
# меняем только GrLivArea, смотрим форму кривой каждой модели.

template_num = X_tr.median(numeric_only=True)
template_cat = X_tr[cat_features].mode().iloc[0]
template = pd.concat([template_num, template_cat]).reindex(X_tr.columns)

grliv_range = np.linspace(500, 4500, 120)
synth = pd.DataFrame([template.values] * len(grliv_range), columns=X_tr.columns)
synth['GrLivArea'] = grliv_range

synth_ridge = synth.copy()
for c in num_features:
    synth_ridge[c] = pd.to_numeric(synth_ridge[c], errors='coerce').fillna(X_tr[c].median())
for c in cat_features:
    synth_ridge[c] = synth_ridge[c].fillna('missing').astype(str)
ridge_curve = ridge.predict(synth_ridge)

synth_cb = synth.copy()
for c in cat_features:
    synth_cb[c] = synth_cb[c].fillna('missing').astype(str)
cb_curve = cat_model.predict(synth_cb)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(grliv_range, np.expm1(ridge_curve), label='Ridge — прямая линия', linewidth=2.5)
ax.plot(grliv_range, np.expm1(cb_curve), label='CatBoost — ступеньки и изломы', linewidth=2.5)
ax.set_xlabel('GrLivArea (жилая площадь, кв.фт)')
ax.set_ylabel('предсказанная цена дома')
ax.set_title('Как модель видит зависимость «площадь → цена»\n(все остальные фичи зафиксированы на медиане/моде)')
ax.legend()
ax.grid(alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))
plt.tight_layout()
plt.savefig('line_vs_function.png', dpi=120, bbox_inches='tight')
plt.show()
print('[ok] сохранил line_vs_function.png')

In [ ]:
# График 3: training curve CatBoost (бонус для семинара)
# Показывает, как RMSE на train и eval ведут себя итерация за итерацией,
# и в какой момент сработал early stopping.

evals = cat_model.get_evals_result()
train_rmse = evals['learn']['RMSE']
eval_rmse = evals['validation']['RMSE']
best_iter = cat_model.get_best_iteration()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_rmse, label='train RMSE', linewidth=2)
ax.plot(eval_rmse, label='eval (test) RMSE', linewidth=2)
ax.axvline(best_iter, color='red', linestyle='--', alpha=0.7,
           label=f'best iteration = {best_iter}')
ax.set_xlabel('итерация бустинга (= номер дерева)')
ax.set_ylabel('RMSE на log(SalePrice)')
ax.set_title('Как CatBoost учится: train ползёт вниз, eval останавливается')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('training_curve.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'[ok] сохранил training_curve.png — обучение остановилось на {best_iter}-й итерации')

## Шаг 5. TODO 2 — SHAP summary plot (решение)

Что разобрать на семинаре:

- `OverallQual` обычно лидер — это та самая «общая оценка качества дома 1—10», которая фактически кодирует то, что человеческий оценщик увидел бы за минуту.
- `GrLivArea` — жилая площадь над землёй. Прямая линия «больше → дороже», на графике точки растягиваются вправо.
- Категориальные фичи (`Neighborhood`) тоже попадают в топ — это та самая фишка CatBoost'а: район «NoRidge» автоматически получает большой плюс, без one-hot и тюнинга.

In [ ]:
import shap

explainer = shap.TreeExplainer(cat_model)
shap_values = explainer.shap_values(X_te_cb)

shap.summary_plot(shap_values, X_te_cb, max_display=10, show=False)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print('[ok] сохранил shap_summary.png')

**Эталонный ответ:** топ-3 фичи, которые сильнее всего двигают цену дома, — это **`OverallQual` (общее качество отделки), `GrLivArea` (жилая площадь над землёй) и `Neighborhood` (район)** — то есть ровно те три вещи, которые и человеческий оценщик назвал бы первыми.

## Шаг 6. TODO 3 — Waterfall plot для самого дорогого дома (решение)

На семинаре полезно показать, что **waterfall — это не «важность вообще», а декомпозиция конкретного предсказания**: вот базовая линия (среднее по тесту), вот что эта конкретная фича добавила или отняла, итого — финальная цена. Это та картинка, которую можно показать оценщику или клиенту.

In [ ]:
preds = cat_model.predict(X_te_cb)
idx = int(np.argmax(preds))  # самый дорогой по предсказанию

shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[idx],
        base_values=explainer.expected_value,
        data=X_te_cb.iloc[idx],
        feature_names=X_te_cb.columns.tolist(),
    ),
    show=False,
)
plt.tight_layout()
plt.savefig('shap_waterfall.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'предсказанная цена: ${np.expm1(preds[idx]):,.0f}')
print(f'настоящая цена:     ${np.expm1(y_te.iloc[idx]):,.0f}')
print()
print('топ-5 фичей этого дома (по абсолютному вкладу в цену):')
contribs = pd.Series(shap_values[idx], index=X_te_cb.columns)
for name, val in contribs.abs().sort_values(ascending=False).head(5).items():
    sign = '+' if contribs[name] > 0 else '−'
    print(f'  {sign} {abs(contribs[name]):.3f}  {name} = {X_te_cb.iloc[idx][name]}')

**Эталонный ответ:** модель оценила этот дом так дорого, потому что у него **`OverallQual=10` (топовое качество), очень большая `GrLivArea`, престижный `Neighborhood` (например, NoRidge или NridgHt) и недавний `YearBuilt`** — каждая из этих фичей на waterfall добавляет к базовой цене положительный кусок.

На семинаре стоит проговорить: **base value ≈ среднее `log(SalePrice)` в обучающей выборке** (около `12.0`, что в долларах ≈ `$163k` — это «средний дом» Эймса), а сумма всех плюсов от фичей довела предсказание до ~`13.0` (≈ `$440k`).

## Чек-лист семинара

Что обязательно показать студентам, чтобы зацепить понимание:

1. **Сравнение Ridge vs CatBoost** в одной табличке. Главный посыл: «20 строк CatBoost'а бьют 40 строк Ridge с препроцессингом — и без feature engineering'а».
2. **`cat_features` ЯВНО.** Удалите параметр, перезапустите — увидите либо `ValueError`, либо RMSE в районе 0.20+. Это лучший способ закрепить «не забывайте».
3. **`early_stopping_rounds` живой.** Покажите training curve — видно, как train RMSE продолжает падать, а eval RMSE разворачивается, и красная пунктирная линия отмечает, где сработал stop. Это не теоретический параметр.
4. **График «линия vs функция»** — главный визуал лекции. Дайте студентам 30 секунд посмотреть и сами скажут «о, ну да, тут прямая, а тут — нет».
5. **SHAP summary plot** — три фичи в топе, объясните почему именно эти.
6. **SHAP waterfall** — конкретный дом, прокомментируйте каждую полоску. Этот скилл переносится буквально на любую задачу в карьере.

Возможные вопросы и ответы:

- *«А почему log(SalePrice), а не сам SalePrice?»* — потому что цены ходят на порядок (с $35k до $755k); ошибка модели для дома за $50k и за $500k не должна быть в одних абсолютных долларах, она должна быть в относительных процентах. Лог-шкала ровно это и даёт.
- *«А что если я хочу AUC?»* — AUC только для бинарной классификации. Для регрессии — RMSE, MAE, R². В лекции есть параграф про дисбаланс — это уже область классификации.
- *«А почему график CatBoost не идеально ступенчатый, а с наклонами?»* — потому что точек 120, а в одной точке мы видим вклад нескольких деревьев одновременно, и они интерполируются. Если приглядеться — видно явные «плато» и «изломы».
- *«А когда вообще не нужен CatBoost?»* — если строк меньше 1000, Ridge или Random Forest часто на одном уровне и проще объясняются (это **trap** «У меня всего 500 строк» из лекции).